# Agentic Multi-Hop RAG — Colab Experiment
**Before running:** `Runtime → Change runtime type → A100 GPU`

In [1]:
# Cell 1: Mount Drive and set project path
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = '/content/drive/MyDrive/CS572/Final'
os.chdir(PROJECT_PATH)

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))


Mounted at /content/drive
Working directory: /content/drive/MyDrive/CS572/Final
Files: ['Final', 'tests', 'src', 'docs', '.git', '.pytest_cache', 'results', '.claude', 'requirements.txt', '.env.example', '.gitignore', 'README.md', 'results.ipynb', '.env', 'colab_experiment.ipynb', 'IR Final Project.pdf']


In [2]:
# Cell 2: Install dependencies
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q -U rank-bm25 datasets sentence-transformers python-dotenv google-genai
print('Done.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 60.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Done.


In [3]:
# Cell 3: Load .env and log in to Hugging Face
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(os.path.join(PROJECT_PATH, '.env'))

hf_token = os.getenv('HF_TOKEN')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not hf_token:
    raise ValueError('Missing HF_TOKEN in .env')
# Gemini is optional. Only needed when RUN_FRONTIER = True below.

login(token=hf_token)

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
if gemini_api_key:
    os.environ['GEMINI_API_KEY'] = gemini_api_key

print('HF_TOKEN loaded:', bool(hf_token))
print('GEMINI_API_KEY loaded:', bool(gemini_api_key))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN loaded: True
GEMINI_API_KEY loaded: True


In [4]:
# Cell 4: Verify GPU
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')


CUDA available: True
GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


In [5]:
# Cell 5: Add project root to Python path and clear stale caches
import sys
import shutil

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

for root, dirs, _ in os.walk(PROJECT_PATH):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Path set and cache cleared.')


Path set and cache cleared.


In [7]:
# Cell 6: Pilot run. Set RUN_FRONTIER=True only if Gemini has available credits.
from src.run_experiment import run

RUN_FRONTIER = True

pilot_results = run(
    num_samples=5,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-flash",
    frontier_num_samples=5,
)

print('Pilot results:', pilot_results)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running baseline RAG on 5 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running Agentic RAG on 5 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running frontier Agentic RAG on 5 samples...

=== RESULTS ===
Metric            Baseline    Agentic   Frontier
------------------------------------------------
EM                  0.0000     0.0000     0.0000
F1                  0.4167     0.5167     0.1000
MRR                 0.2667     0.3000     0.2667
NDCG@10             0.3985     0.4617     0.4069
avg_hops            1.0000     3.8000     2.4000
Pilot results: {'metadata': {'num_samples': 5, 'baseline_provider': 'hf', 'baseline_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'agent_provider': 'hf', 'agent_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'frontier_provider': 'gemini', 'frontier_model_name': 'gemini-2.5-pro', 'frontier_num_samples': 5, 'frontier_completed_samples': 5}, 'baseline': {'EM': 0.0, 'F1': 0.41666666666666663, 'MRR': 0.26666666666666666, 'NDCG@10': 0.3984565739436487, 'avg_hops': 1.0}, 'agentic': {'EM': 0.0, 'F1': 0.5166666666666666, 'MRR': 0.3, 'NDCG@10': 0.4616940730779732, 'avg_hops': 3.8, 'p

In [10]:
# Cell 7: Full run with Gemini frontier ceiling.
from src.run_experiment import run

RUN_FRONTIER = True

results = run(
    num_samples=50,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-flash",
    frontier_num_samples=50,  # adjust higher if budget/time allows
)

print('Final results:', results)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running baseline RAG on 50 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running Agentic RAG on 50 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running frontier Agentic RAG on 50 samples...

=== RESULTS ===
Metric            Baseline    Agentic   Frontier
------------------------------------------------
EM                  0.3200     0.2200     0.1600
F1                  0.4211     0.4166     0.2252
MRR                 0.3137     0.3552     0.3363
NDCG@10             0.3880     0.4728     0.4179
avg_hops            1.0000     3.5200     2.5600
Final results: {'metadata': {'num_samples': 50, 'baseline_provider': 'hf', 'baseline_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'agent_provider': 'hf', 'agent_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'frontier_provider': 'gemini', 'frontier_model_name': 'gemini-2.5-pro', 'frontier_num_samples': 50, 'frontier_completed_samples': 50}, 'baseline': {'EM': 0.32, 'F1': 0.42114285714285715, 'MRR': 0.31366666666666665, 'NDCG@10': 0.38799738340253226, 'avg_hops': 1.0}, 'agentic': {'EM': 0.22, 'F1': 0.4165714285714286, 'MRR': 0.3551594516594517, 'NDCG@10': 0.472831707209883

In [11]:
# Cell 8: Inspect saved aggregate results
import json

results_path = os.path.join(PROJECT_PATH, 'results', 'experiment_results.json')
with open(results_path) as f:
    saved = json.load(f)

print(json.dumps(saved, indent=2))
print('Saved to:', results_path)


{
  "metadata": {
    "num_samples": 50,
    "baseline_provider": "hf",
    "baseline_model_name": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "agent_provider": "hf",
    "agent_model_name": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "frontier_provider": "gemini",
    "frontier_model_name": "gemini-2.5-pro",
    "frontier_num_samples": 50,
    "frontier_completed_samples": 50
  },
  "baseline": {
    "EM": 0.32,
    "F1": 0.42114285714285715,
    "MRR": 0.31366666666666665,
    "NDCG@10": 0.38799738340253226,
    "avg_hops": 1.0
  },
  "agentic": {
    "EM": 0.22,
    "F1": 0.4165714285714286,
    "MRR": 0.3551594516594517,
    "NDCG@10": 0.47283170720988354,
    "avg_hops": 3.52,
    "per_hop_MRR": 0.4490277777777778,
    "per_hop_NDCG@10": 0.5152860388246122
  },
  "frontier_agentic": {
    "EM": 0.16,
    "F1": 0.22523809523809527,
    "MRR": 0.3362698412698413,
    "NDCG@10": 0.41790354990274153,
    "avg_hops": 2.56,
    "per_hop_MRR": 0.3949166666666667,
    "per_hop_NDCG@10":

In [ ]:
# Cell 9: White-box breakdowns and case studies
from src.reporting import (
    load_detailed_results,
    print_support_doc_comparison,
    print_support_doc_breakdown,
    print_retrieval_hop_need_comparison,
    print_case_studies,
    print_intermediate_trace,
)

details = load_detailed_results()

# --- Multi-hop vs single-hop need comparison ---
# support_docs_needed: did the question need >=2 gold docs?
print_support_doc_comparison(details)
# retrieval_hop_need: was a single retrieval pass sufficient to find all gold docs?
print_retrieval_hop_need_comparison(details)

# --- Per-system breakdowns ---
print_support_doc_breakdown(details, system_name="baseline")
print_support_doc_breakdown(details, system_name="agentic")
if "frontier_agentic" in details:
    print_support_doc_breakdown(details, system_name="frontier_agentic")

# --- Case studies: agentic ---
print_case_studies(details, system_name="agentic", group_name="successful_multi_doc")
print_case_studies(details, system_name="agentic", group_name="failed_multi_doc")

# --- Case studies: frontier (same question slice for apples-to-apples comparison) ---
if "frontier_agentic" in details:
    print_case_studies(details, system_name="frontier_agentic", group_name="successful_multi_doc")
    print_case_studies(details, system_name="frontier_agentic", group_name="failed_multi_doc")

# --- Intermediate agent traces (hop-by-hop decision log) ---
print_intermediate_trace(details, system_name="agentic", trace_index=0)
print_intermediate_trace(details, system_name="agentic", trace_index=1)
if "frontier_agentic" in details:
    print_intermediate_trace(details, system_name="frontier_agentic", trace_index=0)


In [ ]:
# Cell 10: Frontier model ceiling
# Gemini-2.5-Flash is the frontier comparison model — if the open-source agentic model
# matches or beats it, the bottleneck is the retrieval loop, not model capacity.
from src.reporting import load_detailed_results, print_frontier_ceiling_summary

details = load_detailed_results()
print_frontier_ceiling_summary(details)


In [ ]:
# Cell 11: White-box traces — full retrieved text at each hop
# Shows WHY the agent succeeded or failed: what text was actually retrieved.
from src.reporting import load_detailed_results, print_full_trace_with_text

details = load_detailed_results()

agentic_traces = details["agentic"]["traces"]
multi_success = [t for t in agentic_traces if t["requires_multiple_documents"] and t["exact_match"]]
multi_fail    = [t for t in agentic_traces if t["requires_multiple_documents"] and not t["exact_match"]]

print("\n" + "="*70)
print("WHITE-BOX: Successful multi-hop (Agentic)")
print("="*70)
if multi_success:
    print_full_trace_with_text(multi_success[0])

print("\n" + "="*70)
print("WHITE-BOX: Failed multi-hop (Agentic)")
print("="*70)
if multi_fail:
    print_full_trace_with_text(multi_fail[0])

if "frontier_agentic" in details:
    frontier_traces = details["frontier_agentic"]["traces"]
    f_success = [t for t in frontier_traces if t["requires_multiple_documents"] and t["exact_match"]]
    f_fail    = [t for t in frontier_traces if t["requires_multiple_documents"] and not t["exact_match"]]

    print("\n" + "="*70)
    print("WHITE-BOX: Successful multi-hop (Frontier — Gemini-2.5-Flash)")
    print("="*70)
    if f_success:
        print_full_trace_with_text(f_success[0])

    print("\n" + "="*70)
    print("WHITE-BOX: Failed multi-hop (Frontier — Gemini-2.5-Flash)")
    print("="*70)
    if f_fail:
        print_full_trace_with_text(f_fail[0])


In [13]:
# Cell 12: Raw trace inspection (single sample) — manual deep dive
details = load_detailed_results()
trace = details["agentic"]["traces"][0]

print("Question:", trace["question"])
print("Gold:", trace["gold_answer"])
print("Pred:", trace["predicted_answer"])
print("Sub-queries:", trace["sub_queries"])
print("Requires multiple docs:", trace["requires_multiple_documents"])
print("Num hops:", trace["num_hops"])

for hop in trace["per_hop"]:
    print("\nHop query:", hop["query"])
    for doc in hop["docs"]:
        print("-", doc["title"], "|", doc["text_preview"])


Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: No, different nationalities.
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', "Scott Derrickson's birthplace", "Ed Wood's birthplace", "Ed Wood's nationality"]
Requires multiple docs: True
Num hops: 4

Hop query: Were Scott Derrickson and Ed Wood of the same nationality?
- Tyler Bates | Tyler Bates. Tyler Bates (born June 5, 1965) is an American musician, music producer, and composer for films, television, and video games.  Much of his work is in the action and horror film genres, with films like "Dawn 
- Woodson, Arkansas | Woodson, Arkansas. Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North Little Rock–Conway Metropo
- Doctor Strange (2016 film) | Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero f